In [1]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd

# ===== 路径配置 =====
DATA_ROOT = Path("/mydata/doc2validate/data")
DOWNLOADED_ARTIFACTS_DIR = DATA_ROOT / "downloaded_artifacts"

RUN_ROOT = Path("/mydata/doc2validate/results/runs/scidata_4293")
ANALYSIS_DIR = RUN_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v2.csv"
OUTPUT_XLSX = ANALYSIS_DIR / "downloaded_artifact_pool_audit_v2.xlsx"

print("Downloaded artifacts dir:", DOWNLOADED_ARTIFACTS_DIR)
print("Exists:", DOWNLOADED_ARTIFACTS_DIR.exists())
print("Output:", OUTPUT_CSV)

Downloaded artifacts dir: /mydata/doc2validate/data/downloaded_artifacts
Exists: True
Output: /mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v2.csv


In [2]:
MANIFEST_NAME = "ARTIFACT_DOWNLOAD_MANIFEST.json"

TABULAR_SUFFIXES = {
    ".csv", ".tsv", ".txt", ".xlsx", ".xls", ".parquet", ".json", ".jsonl"
}

STRICT_TABULAR_SUFFIXES = {
    ".csv", ".tsv", ".xlsx", ".xls", ".parquet"
}

ARCHIVE_SUFFIXES = {
    ".zip", ".tar", ".gz", ".tgz", ".tar.gz", ".7z", ".rar", ".bz2", ".xz"
}

KNOWN_PROVIDER_DIRS = {
    "github", "zenodo", "direct", "figshare", "osf", "generic", "dataverse"
}


def safe_load_json(path: Path):
    if not path.exists():
        return None

    for enc in ["utf-8", "utf-8-sig"]:
        try:
            with path.open("r", encoding=enc) as f:
                return json.load(f)
        except Exception:
            pass

    try:
        return {"_load_error": path.read_text(errors="replace")[:500]}
    except Exception as e:
        return {"_load_error": str(e)}


def is_hidden_or_system(path: Path) -> bool:
    name = path.name
    return (
        name.startswith(".")
        or name == "__MACOSX"
        or name == ".DS_Store"
        or name == "__pycache__"
    )


def suffix_of(path: Path) -> str:
    name = path.name.lower()
    if name.endswith(".tar.gz"):
        return ".tar.gz"
    return path.suffix.lower()


def is_archive(path: Path) -> bool:
    return suffix_of(path) in ARCHIVE_SUFFIXES


def is_tabular_like(path: Path) -> bool:
    return suffix_of(path) in TABULAR_SUFFIXES


def is_strict_tabular(path: Path) -> bool:
    return suffix_of(path) in STRICT_TABULAR_SUFFIXES


def iter_real_files(article_dir: Path):
    for p in article_dir.rglob("*"):
        if is_hidden_or_system(p):
            continue
        if p.is_file() and p.name != MANIFEST_NAME:
            yield p


def top_level_entries_except_manifest(article_dir: Path):
    entries = []
    for p in article_dir.iterdir():
        if is_hidden_or_system(p):
            continue
        if p.name == MANIFEST_NAME:
            continue
        entries.append(p)
    return entries


def is_under_extracted(article_dir: Path, path: Path) -> bool:
    rel_parts = path.relative_to(article_dir).parts
    return "extracted" in rel_parts


def is_under_archive_dir(article_dir: Path, path: Path) -> bool:
    rel_parts = path.relative_to(article_dir).parts
    return "archive" in rel_parts


def manifest_summary(manifest):
    if not isinstance(manifest, dict):
        return {}

    resources = manifest.get("resources", []) or []

    statuses = Counter()
    handlers = Counter()
    modes = Counter()
    urls = []
    reasons = Counter()

    for r in resources:
        statuses[str(r.get("status", ""))] += 1
        handlers[str(r.get("handler", ""))] += 1
        modes[str(r.get("download_mode", ""))] += 1

        if r.get("url"):
            urls.append(r.get("url"))

        if r.get("reason"):
            reasons[str(r.get("reason"))] += 1

    return {
        "manifest_resource_count": len(resources),
        "manifest_statuses": dict(statuses),
        "manifest_handlers": dict(handlers),
        "manifest_download_modes": dict(modes),
        "manifest_urls": urls,
        "manifest_reasons": dict(reasons),
        "manifest_any_success": any(str(r.get("status")) == "success" for r in resources),
        "manifest_any_skipped": any(str(r.get("status")) == "skipped" for r in resources),
    }


def compact_dict(d):
    if not d:
        return ""
    return json.dumps(d, ensure_ascii=False, sort_keys=True)


def compact_list(xs, max_items=20):
    if not xs:
        return ""

    xs = [str(x) for x in xs]

    if len(xs) > max_items:
        return "; ".join(xs[:max_items]) + f"; ... (+{len(xs) - max_items})"

    return "; ".join(xs)

In [3]:
article_dirs = sorted([p for p in DOWNLOADED_ARTIFACTS_DIR.iterdir() if p.is_dir()])
print("Article dirs:", len(article_dirs))

rows = []

for article_dir in article_dirs:
    article_id = article_dir.name

    manifest_path = article_dir / MANIFEST_NAME
    manifest = safe_load_json(manifest_path)
    ms = manifest_summary(manifest)

    top_entries = top_level_entries_except_manifest(article_dir)
    top_dirs = [p.name for p in top_entries if p.is_dir()]
    top_files = [p.name for p in top_entries if p.is_file()]

    real_files = list(iter_real_files(article_dir))
    real_dirs = [
        p for p in article_dir.rglob("*")
        if p.is_dir() and not is_hidden_or_system(p)
    ]

    archive_files = [p for p in real_files if is_archive(p)]
    archive_dir_files = [p for p in real_files if is_under_archive_dir(article_dir, p)]

    extracted_files = [
        p for p in real_files
        if is_under_extracted(article_dir, p)
    ]

    non_archive_non_manifest_files = [
        p for p in real_files
        if not is_under_archive_dir(article_dir, p)
    ]

    tabular_files = [p for p in real_files if is_tabular_like(p)]
    strict_tabular_files = [p for p in real_files if is_strict_tabular(p)]

    extracted_tabular_files = [
        p for p in extracted_files
        if is_tabular_like(p)
    ]

    extracted_strict_tabular_files = [
        p for p in extracted_files
        if is_strict_tabular(p)
    ]

    non_archive_tabular_files = [
        p for p in non_archive_non_manifest_files
        if is_tabular_like(p)
    ]

    non_archive_strict_tabular_files = [
        p for p in non_archive_non_manifest_files
        if is_strict_tabular(p)
    ]

    file_suffix_counts = Counter(suffix_of(p) or "[no_suffix]" for p in real_files)
    extracted_suffix_counts = Counter(suffix_of(p) or "[no_suffix]" for p in extracted_files)
    non_archive_suffix_counts = Counter(
        suffix_of(p) or "[no_suffix]" for p in non_archive_non_manifest_files
    )

    provider_dirs_present = sorted([
        d for d in top_dirs
        if d.lower() in KNOWN_PROVIDER_DIRS
    ])

    # 顶层是否只有 manifest
    if len(top_entries) == 0:
        artifact_presence = "manifest_only"
    elif len(real_files) == 0:
        artifact_presence = "dirs_but_no_files"
    else:
        artifact_presence = "has_real_artifact"

    has_extracted_content = len(extracted_files) > 0
    has_non_archive_content = len(non_archive_non_manifest_files) > 0
    has_archive_backup = len(archive_files) > 0 or len(archive_dir_files) > 0

    # v2 核心分类：以 extracted / non-archive 内容为准，不把 archive backup 当成问题
    if artifact_presence != "has_real_artifact":
        downloaded_pool_status_v2 = "exclude_no_downloaded_content"

    elif len(extracted_strict_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_extracted_strict_tabular_candidate"

    elif len(extracted_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_extracted_tabular_like_candidate"

    elif len(non_archive_strict_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_non_archive_strict_tabular_candidate"

    elif len(non_archive_tabular_files) > 0:
        downloaded_pool_status_v2 = "has_non_archive_tabular_like_candidate"

    elif has_extracted_content:
        downloaded_pool_status_v2 = "has_extracted_non_tabular_content"

    elif has_archive_backup and not has_non_archive_content:
        downloaded_pool_status_v2 = "archive_backup_only_no_extracted_content"

    else:
        downloaded_pool_status_v2 = "has_non_archive_non_tabular_content"

    rows.append({
        "article_id": article_id,
        "article_dir": str(article_dir),

        "manifest_exists": manifest_path.exists(),
        "artifact_presence": artifact_presence,
        "downloaded_pool_status_v2": downloaded_pool_status_v2,

        "top_level_entry_count_except_manifest": len(top_entries),
        "top_level_dirs_except_manifest": compact_list(top_dirs),
        "top_level_files_except_manifest": compact_list(top_files),
        "provider_dirs_present": compact_list(provider_dirs_present),

        "has_archive_backup": has_archive_backup,
        "has_extracted_content": has_extracted_content,
        "has_non_archive_content": has_non_archive_content,

        "real_file_count_except_manifest": len(real_files),
        "real_dir_count": len(real_dirs),

        "archive_file_count": len(archive_files),
        "archive_dir_file_count": len(archive_dir_files),

        "extracted_file_count": len(extracted_files),
        "non_archive_file_count": len(non_archive_non_manifest_files),

        "tabular_like_file_count_all": len(tabular_files),
        "strict_tabular_file_count_all": len(strict_tabular_files),

        "extracted_tabular_like_file_count": len(extracted_tabular_files),
        "extracted_strict_tabular_file_count": len(extracted_strict_tabular_files),

        "non_archive_tabular_like_file_count": len(non_archive_tabular_files),
        "non_archive_strict_tabular_file_count": len(non_archive_strict_tabular_files),

        "file_suffix_counts_all": compact_dict(dict(file_suffix_counts)),
        "extracted_suffix_counts": compact_dict(dict(extracted_suffix_counts)),
        "non_archive_suffix_counts": compact_dict(dict(non_archive_suffix_counts)),

        "sample_extracted_files": compact_list(
            [str(p.relative_to(article_dir)) for p in extracted_files[:30]],
            max_items=30
        ),
        "sample_extracted_tabular_files": compact_list(
            [str(p.relative_to(article_dir)) for p in extracted_tabular_files[:20]],
            max_items=20
        ),
        "sample_non_archive_files": compact_list(
            [str(p.relative_to(article_dir)) for p in non_archive_non_manifest_files[:30]],
            max_items=30
        ),

        "manifest_resource_count": ms.get("manifest_resource_count", 0),
        "manifest_any_success": ms.get("manifest_any_success", False),
        "manifest_any_skipped": ms.get("manifest_any_skipped", False),
        "manifest_statuses": compact_dict(ms.get("manifest_statuses")),
        "manifest_handlers": compact_dict(ms.get("manifest_handlers")),
        "manifest_download_modes": compact_dict(ms.get("manifest_download_modes")),
        "manifest_reasons": compact_dict(ms.get("manifest_reasons")),
        "manifest_urls": compact_list(ms.get("manifest_urls", []), max_items=10),
    })

df = pd.DataFrame(rows)
df.head()

Article dirs: 187


,article_id,article_dir,manifest_exists,artifact_presence,downloaded_pool_status_v2,top_level_entry_count_except_manifest,top_level_dirs_except_manifest,top_level_files_except_manifest,provider_dirs_present,has_archive_backup,...,sample_extracted_tabular_files,sample_non_archive_files,manifest_resource_count,manifest_any_success,manifest_any_skipped,manifest_statuses,manifest_handlers,manifest_download_modes,manifest_reasons,manifest_urls
0,s41597-019-0021-x,/mydata/doc2validate/data/downloaded_artifacts...,True,has_real_artifact,has_non_archive_strict_tabular_candidate,1,github,,github,False,...,,github/github.com__nheeren_material_intensity_...,1,False,False,"{""downloaded"": 1}","{""github_handler"": 1}","{""github_archive_zip_then_git_clone"": 1}","{""github_repo_contains_data_artifacts"": 1}",https://github.com/nheeren/material_intensity_db
1,s41597-019-0035-4,/mydata/doc2validate/data/downloaded_artifacts...,True,has_real_artifact,has_extracted_strict_tabular_candidate,1,github,,github,True,...,github/github.com__oesteban_mriqc-webapi-snaps...,github/github.com__oesteban_mriqc-webapi-snaps...,3,False,True,"{""downloaded"": 1, ""skipped"": 2}","{""generic"": 2, ""github_handler"": 1}","{""github_archive_zip"": 1, ""unsupported_landing...","{""github_repo_contains_data_artifacts"": 1, ""un...",https://figshare.com/articles/MRIQC_WebAPI_-_D...
2,s41597-019-0098-2,/mydata/doc2validate/data/downloaded_artifacts...,True,has_real_artifact,has_extracted_strict_tabular_candidate,1,github,,github,True,...,github/github.com__genome-in-a-bottle_giab_dat...,github/github.com__genome-in-a-bottle_giab_dat...,1,False,True,"{""skipped"": 1}","{""github_handler"": 1}","{""github_archive_zip"": 1}","{""no_data_artifacts_found_in_github_repo"": 1}",https://github.com/genome-in-a-bottle/giab_dat...
3,s41597-019-0213-4,/mydata/doc2validate/data/downloaded_artifacts...,True,manifest_only,exclude_no_downloaded_content,0,,,,False,...,,,1,False,True,"{""skipped"": 1}","{""generic"": 1}","{""unsupported_landing_page"": 1}","{""unsupported_or_landing_page_only"": 1}",https://immport.niaid.nih.gov/home
4,s41597-019-0342-9,/mydata/doc2validate/data/downloaded_artifacts...,True,has_real_artifact,has_extracted_tabular_like_candidate,1,github,,github,True,...,github/github.com__practikpharma_PGxCorpus/ext...,github/github.com__practikpharma_PGxCorpus/ext...,3,False,True,"{""downloaded"": 1, ""skipped"": 2}","{""generic"": 2, ""github_handler"": 1}","{""github_archive_zip"": 1, ""unsupported_landing...","{""github_repo_contains_data_artifacts"": 1, ""un...",https://github.com/practikpharma/PGxCorpus; ht...


In [4]:
summary_presence = df["artifact_presence"].value_counts(dropna=False)
summary_status_v2 = df["downloaded_pool_status_v2"].value_counts(dropna=False)

print("Artifact presence:")
display(summary_presence.to_frame("count"))

print("\nDownloaded pool status v2:")
display(summary_status_v2.to_frame("count"))

print("\nTotal article dirs:", len(df))
print("Has real artifact:", (df["artifact_presence"] == "has_real_artifact").sum())
print("Has extracted content:", df["has_extracted_content"].sum())
print("Has extracted strict tabular:", (df["extracted_strict_tabular_file_count"] > 0).sum())
print("Has extracted tabular-like:", (df["extracted_tabular_like_file_count"] > 0).sum())
print("Archive backup only, no extracted content:", (df["downloaded_pool_status_v2"] == "archive_backup_only_no_extracted_content").sum())

Artifact presence:


,count
has_real_artifact,113
manifest_only,56
dirs_but_no_files,18



Downloaded pool status v2:


,count
exclude_no_downloaded_content,74
has_extracted_strict_tabular_candidate,72
has_extracted_non_tabular_content,18
has_extracted_tabular_like_candidate,13
has_non_archive_strict_tabular_candidate,6
has_non_archive_non_tabular_content,2
archive_backup_only_no_extracted_content,2



Total article dirs: 187
Has real artifact: 113
Has extracted content: 103
Has extracted strict tabular: 72
Has extracted tabular-like: 85
Archive backup only, no extracted content: 2


In [5]:
df_has_artifact = df[df["artifact_presence"] == "has_real_artifact"].copy()

display(df_has_artifact[[
    "article_id",
    "downloaded_pool_status_v2",
    "provider_dirs_present",
    "has_extracted_content",
    "extracted_file_count",
    "extracted_strict_tabular_file_count",
    "extracted_tabular_like_file_count",
    "non_archive_strict_tabular_file_count",
    "file_suffix_counts_all",
    "extracted_suffix_counts",
]].head(30))

,article_id,downloaded_pool_status_v2,provider_dirs_present,has_extracted_content,extracted_file_count,extracted_strict_tabular_file_count,extracted_tabular_like_file_count,non_archive_strict_tabular_file_count,file_suffix_counts_all,extracted_suffix_counts
0,s41597-019-0021-x,has_non_archive_strict_tabular_candidate,github,False,0,0,0,3,"{"".2"": 1, "".bib"": 1, "".csv"": 3, "".idx"": 1, "".m...",
1,s41597-019-0035-4,has_extracted_strict_tabular_candidate,github,True,21,7,12,7,"{"".bson"": 4, "".csv"": 7, "".idx"": 1, "".json"": 5,...","{"".bson"": 4, "".csv"": 7, "".json"": 5, "".png"": 1,..."
2,s41597-019-0098-2,has_extracted_strict_tabular_candidate,github,True,141,3,3,3,"{"".0_grch37_grch38_06202016"": 1, "".1_grch37_gr...","{"".0_grch37_grch38_06202016"": 1, "".1_grch37_gr..."
4,s41597-019-0342-9,has_extracted_tabular_like_candidate,github,True,56,0,16,0,"{"".ddg_tree_comp_reps"": 1, "".jar"": 1, "".lua"": ...","{"".ddg_tree_comp_reps"": 1, "".jar"": 1, "".lua"": ..."
6,s41597-020-00609-9,has_extracted_strict_tabular_candidate,github,True,34,8,9,8,"{"".bib"": 1, "".csv"": 8, "".docx"": 1, "".json"": 1,...","{"".bib"": 1, "".csv"": 8, "".docx"": 1, "".json"": 1,..."
7,s41597-020-00610-2,has_extracted_strict_tabular_candidate,github,True,10,4,5,4,"{"".csv"": 4, "".md"": 1, "".pdf"": 3, "".txt"": 1, ""....","{"".csv"": 4, "".md"": 1, "".pdf"": 3, "".txt"": 1, ""[..."
10,s41597-020-00676-y,has_extracted_strict_tabular_candidate,github,True,17,8,8,8,"{"".csv"": 7, "".md"": 4, "".pdf"": 2, "".rmd"": 1, ""....","{"".csv"": 7, "".md"": 4, "".pdf"": 2, "".rmd"": 1, ""...."
11,s41597-020-00682-0,has_extracted_strict_tabular_candidate,github,True,78,30,31,30,"{"".cff"": 1, "".csv"": 30, "".md"": 1, "".png"": 3, ""...","{"".cff"": 1, "".csv"": 30, "".md"": 1, "".png"": 3, ""..."
12,s41597-020-00688-8,has_extracted_strict_tabular_candidate,github,True,2235,1613,1706,1613,"{"".bat"": 1, "".csv"": 1485, "".html"": 4, "".ico"": ...","{"".bat"": 1, "".csv"": 1485, "".html"": 4, "".ico"": ..."
13,s41597-020-00702-z,has_extracted_strict_tabular_candidate,github,True,137,4,5,4,"{"".css"": 3, "".csv"": 4, "".html"": 30, "".jpg"": 6,...","{"".css"": 3, "".csv"": 4, "".html"": 30, "".jpg"": 6,..."


In [6]:
df_no_content = df[df["artifact_presence"] != "has_real_artifact"].copy()

display(df_no_content[[
    "article_id",
    "artifact_presence",
    "downloaded_pool_status_v2",
    "manifest_statuses",
    "manifest_handlers",
    "manifest_download_modes",
    "manifest_reasons",
    "manifest_urls",
]].head(30))

,article_id,artifact_presence,downloaded_pool_status_v2,manifest_statuses,manifest_handlers,manifest_download_modes,manifest_reasons,manifest_urls
3,s41597-019-0213-4,manifest_only,exclude_no_downloaded_content,"{""skipped"": 1}","{""generic"": 1}","{""unsupported_landing_page"": 1}","{""unsupported_or_landing_page_only"": 1}",https://immport.niaid.nih.gov/home
5,s41597-020-00599-8,manifest_only,exclude_no_downloaded_content,"{""skipped"": 1}","{""generic"": 1}","{""unsupported_landing_page"": 1}","{""unsupported_or_landing_page_only"": 1}",https://uploads.users.earthengine.app/view/hab...
8,s41597-020-00624-w,manifest_only,exclude_no_downloaded_content,"{""skipped"": 1}","{""generic"": 1}","{""unsupported_landing_page"": 1}","{""unsupported_or_landing_page_only"": 1}",https://www.pangaea.de
9,s41597-020-00662-4,manifest_only,exclude_no_downloaded_content,"{""skipped"": 2}","{""generic"": 2}","{""unsupported_landing_page"": 2}","{""unsupported_or_landing_page_only"": 2}",https://figshare.com/articles/A_2015_inventory...
18,s41597-020-0360-7,dirs_but_no_files,exclude_no_downloaded_content,"{""failed"": 1, ""skipped"": 1}","{""generic"": 1, ""github_handler"": 1}","{""github_archive_zip_then_git_clone"": 1, ""unsu...","{""Command '['git', 'clone', '--depth', '1', 'h...",https://github.com/CRazorback/The-SUSTech-SYSU...
19,s41597-020-0379-9,manifest_only,exclude_no_downloaded_content,"{""skipped"": 2}","{""generic"": 2}","{""unsupported_landing_page"": 2}","{""unsupported_or_landing_page_only"": 2}",https://springernature.figshare.com/collection...
21,s41597-020-0448-0,dirs_but_no_files,exclude_no_downloaded_content,"{""skipped"": 2}","{""generic"": 1, ""github_handler"": 1}","{""github_archive_zip"": 1, ""unsupported_landing...","{""repo_too_large_precheck"": 1, ""unsupported_or...",http://www.healthmap.org/ncov2019; https://git...
26,s41597-021-00905-y,manifest_only,exclude_no_downloaded_content,"{""skipped"": 1}","{""generic"": 1}","{""unsupported_landing_page"": 1}","{""unsupported_or_landing_page_only"": 1}",https://figshare.com/collections/Gene-Rare_Dis...
29,s41597-021-00935-6,dirs_but_no_files,exclude_no_downloaded_content,"{""skipped"": 2}","{""direct_file"": 2}","{""compressed_archive"": 2}","{""file_too_large"": 2}",https://ddbj.nig.ac.jp/public/ddbj_database/wg...
31,s41597-021-00950-7,dirs_but_no_files,exclude_no_downloaded_content,"{""skipped"": 1}","{""github_handler"": 1}","{""github_archive_zip"": 1}","{""repo_too_large_precheck"": 1}",https://github.com/asjadnaqvi/COVID19-European...


In [7]:
df.to_csv(OUTPUT_CSV, index=False)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name="all_downloaded_dirs")
    df_has_artifact.to_excel(writer, index=False, sheet_name="has_real_artifact")
    df_no_content.to_excel(writer, index=False, sheet_name="no_real_artifact")
    summary_presence.to_frame("count").to_excel(writer, sheet_name="summary_presence")
    summary_status_v2.to_frame("count").to_excel(writer, sheet_name="summary_status_v2")

print("Saved:")
print(OUTPUT_CSV)
print(OUTPUT_XLSX)

Saved:
/mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v2.csv
/mydata/doc2validate/results/runs/scidata_4293/analysis/downloaded_artifact_pool_audit_v2.xlsx
